In [2]:
import numpy as np

# Database of 5 vectors (dim = 3)
vectors = np.array([
    [1, 0, 0],
    [0, 1, 0],
    [0.9, 0.1, 0],
    [0, 0, 1],
    [0.2, 0.8, 0]
])

# Query vector
query = np.array([0, 0, 1])

# Cosine similarity
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)) # norm: measure the size or length of a vector/matrix 

# Compute similarity for each vector
similarities = [cosine_similarity(query, vec) for vec in vectors]

# Get top 3 most similar
top_k = np.argsort(similarities)[-3:][::-1]
print("Top 3 indexes:", top_k)


Top 3 indexes: [3 4 2]


## Data Structure for retrival 

#### Brute-force Search (Linear Scan)


How it works: Compute similarity between query and all vectors

Time Complexity: O(N·d), where N = number of vectors, d = dimensions

Pros:

- Simple

- Always accurate (no approximation)

Cons:

- Very slow for large datasets


In [3]:
from numpy.linalg import norm

def cosine_similarity(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

# Find top-2 most similar vectors
sims = [cosine_similarity(query, v) for v in vectors]
top_indices = np.argsort(sims)[-2:][::-1]

print("Brute-force top matches:", top_indices)


Brute-force top matches: [3 4]


## Tree-Based Methods
a) KD-Tree (k-dimensional tree)

- Idea: Recursively split the data by dimensions to build a binary tree

- Each node splits space into two halves (e.g., by x, y, z…)

- Best for: low-dimensional data (≤ 20D)

- Time Complexity: O(log N) in ideal cases

- Problem: For high dimensions (>30), it degrades close to brute-force → the curse of dimensionality

b) Ball Tree

- Similar to KD-tree but uses hyperspheres instead of axis-aligned cuts

- Handles skewed or clustered data better than KD-trees

- Still suffers in high dimensions

In [6]:
#KD-Tree for efficient nearest neighbor search
from sklearn.neighbors import KDTree

tree = KDTree(vectors, leaf_size=2)
dist, ind = tree.query([query], k=2)

print("KD-Tree top matches:", ind[0])


KD-Tree top matches: [3 4]


In [ ]:
# Ball Tree for efficient nearest neighbor search
from sklearn.neighbors import BallTree

ball_tree = BallTree(vectors, leaf_size=2, metric='euclidean') # Using 'euclidean' metric for BallTree
dist, ind = ball_tree.query([query], k=2)

print("Ball Tree top matches:", ind[0])


Ball Tree top matches: [3 4]


## Graph-Based Methods: HNSW
#### ⚡ HNSW (Hierarchical Navigable Small World)
- Build a graph where:

    - Vectors are nodes

    - Edges link each node to its nearest neighbors

- During search:

    - Traverse the graph from an entry point

    - Follow the edges toward nodes closer to the query

- Hierarchical layers improve speed

- Very fast and accurate (used in production systems)

Pros:

- Excellent performance even with millions of vectors

- Supports incremental updates (insert/delete)

Cons:

- More complex to implement

- Uses more memory

In [8]:
import hnswlib

dim = 3
num_elements = len(vectors)

# Initialize index
index = hnswlib.Index(space='cosine', dim=dim)
index.init_index(max_elements=10, ef_construction=100, M=16)

# Add vectors
index.add_items(vectors)

# Search
labels, distances = index.knn_query(query, k=2)

print("HNSW top matches:", labels[0])


HNSW top matches: [3 1]


## Quantization-Based Methods
Used to compress vectors and speed up search.

### a) Product Quantization (PQ)

- Break each vector into parts (subvectors)

- Quantize (round off) each subvector into codebooks

- Search happens in compressed space using approximated distances

### b) IVF (Inverted File Index)

- Divide the vector space into clusters

- Store vectors into their closest cluster

- At search time: search only the closest clusters (not all vectors)

🔁 Often combined: IVF + PQ = fast & memory-efficient ANN

In [9]:
import faiss

# Convert data to float32
xb = vectors.astype('float32')
xq = np.array([query], dtype='float32')

# Create index: IVF with 2 clusters
quantizer = faiss.IndexFlatL2(3)  # base index
index = faiss.IndexIVFFlat(quantizer, 3, 2)  # IVF

# Train + add
index.train(xb)
index.add(xb)

# Search
index.nprobe = 1
distances, indices = index.search(xq, 2)

print("IVF+PQ top matches:", indices[0])


IVF+PQ top matches: [3 4]
